# Confidence (uncertainty) maps for seg-model v1 (M3-T04)

Every prediction of the selected model (nnU-Net, `models/registry/seg-model/v1`) gets a map of where the model is unsure.
The model has no dropout, so the scan is predicted **8 times, once per mirror flip** (nnU-Net's own test-time
augmentation, run pass by pass), and the passes are compared. The mask stays exactly the seg-model v1 mask.

On the **188 validation patients** the notebook also checks that the map is useful: calibration, whether uncertain
voxels are the wrong ones, and whether uncertain patients have lower Dice. Locked test patients are never loaded.

**How to run:** **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About 1–3 hours for all 188
patients. If Colab stops, just **Run all** again: it continues where it stopped. Results: `BrainMRI_Data/runs/M3-T04_uncertainty/`.

In [ ]:
# 1. Settings, GPU and Drive
BRANCH = "main"
CASES = None          # None = all 188 validation patients; a number = only the first N (quick check)
SAVE_MAPS = 10        # NIfTI maps (mask, uncertainty, disagreement) are written for the first N patients
MAX_HOURS = 3.5       # stops cleanly after this long; Run all again to continue

import os, sys, json, hashlib, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
CLEANED_TAR = BASE / "processed/brats2021_trainval_v1.tar"
MODEL_DIR = BASE / "runs/M2-T04_nnunet/nnUNet_results/Dataset501_BraTS2021/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres"
OUT = BASE / "runs/M3-T04_uncertainty"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: change the runtime type to T4 GPU"

In [ ]:
# 2. Code from the team repo, and the packages (nnU-Net version from the model manifest)
if Path("/content/repo/.git").exists():
    !git -C /content/repo fetch -q origin && git -C /content/repo checkout -q {BRANCH} && git -C /content/repo pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/nhahub/NHA-5-187.git /content/repo
!git -C /content/repo log --oneline -1
!pip -q install "nnunetv2==2.8.1" "monai>=1.4"
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):   # nnU-Net warns when these are unset
    os.environ.setdefault(k, "/content/nnunet_unused")
sys.path.insert(0, "/content/repo/src")

In [ ]:
# 3. The model: check the checkpoint is the registered seg-model v1 (SHA-256 from manifest.json)
manifest = json.load(open("/content/repo/models/registry/seg-model/v1/manifest.json"))
ckpt = MODEL_DIR / "fold_0" / manifest["nnunet"]["checkpoint"]
h = hashlib.sha256()
with open(ckpt, "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        h.update(block)
assert h.hexdigest() == manifest["checkpoint_file"]["sha256"], "checkpoint differs from the registered seg-model v1"
print("checkpoint OK:", ckpt.name, f"{ckpt.stat().st_size / 1e6:.0f} MB")

In [ ]:
# 4. Cleaned BraTS 2021 training + validation patients (restored from the Drive archive, about 10 minutes)
processed = Path("/content/processed/brats2021")
if not processed.exists():
    !mkdir -p /content/processed && tar -xf "{CLEANED_TAR}" -C /content/processed
print(len(list(processed.glob("BraTS2021_*"))), "cleaned patients ready")

In [ ]:
# 5. Confidence maps + checks on the validation patients (continues where a previous session stopped)
%cd /content/repo
cases_arg = f"--cases {CASES}" if CASES else ""
!python scripts/uncertainty.py --model-dir "{MODEL_DIR}" --out "{OUT}" --save-maps {SAVE_MAPS} --max-hours {MAX_HOURS} {cases_arg}

In [ ]:
# 6. Results
from IPython.display import Image, display
if (OUT / "summary.json").exists():
    print(json.dumps(json.load(open(OUT / "summary.json")), indent=1))
    for png in sorted((OUT / "figures").glob("*.png")):
        display(Image(str(png), width=1000))
else:
    print("not finished yet: Run all again to continue")

In [ ]:
# 7. Finish writing to Drive
drive.flush_and_unmount()